# AfriVoices — P0b : dev STRATIFIÉ scripted/unscripted, recalibration, grille par (langue × type)

La sonde du test a révélé que chaque clip porte une colonne **`type`** (scripted / Unscripted).
Ce notebook en tire tout ce qui est tirable sans réentraîner :

1. **§4 — composition EXACTE du test** : lecture des colonnes `language`/`type` des 94
   parquets (sans l'audio, rapide) → matrice langue × type sur les 41 733 clips.
2. **§5 — dev stratifié** : ~80 clips par (langue × type) depuis `anvke/` (les noms de
   parquets portent déjà scripted/unscripted), **toutes durées** — le fenêtrage est réglé
   (décodage direct gagnant), les clips longs passent en direct avec secours fp16 si OOM.
   swa : case « mixte » depuis `prepared_v5` (pas de brut swa sur le Drive) — caveat affiché.
3. **§7 — recalibration** : WER par case au réglage actuel (0.7, 0.5), puis projection
   leaderboard **pondérée par les poids exacts du test**. À comparer à 0.40283 : si le
   résiduel est petit, le mystère du +0.10 est fermé quantitativement.
4. **§8 — grille (α, β) PAR CASE** + comparaison chiffrée de trois politiques
   (tout-(0.7,0.5) / tout-(0.5,0.0) / par-case-adoptées) avec les vrais poids du test,
   et impression du dict `AB` prêt à coller dans `afrivoices_soumission_par_type.ipynb`.

**Session GPU (T4 suffit).** §1 (install → REDÉMARRER → relance §1) → §2 → ... → §8.
Durées : §4 ~3-5 min, §5 ~10-20 min (Drive), §6 ~25-40 min (GPU), §8 ~30-60 min (CPU).

## 1. Dépendances (redémarrer le runtime après la 1ère exécution)

In [ ]:
import importlib, subprocess, sys
need = [m for m in ["kenlm", "pyctcdecode", "jiwer"] if importlib.util.find_spec(m) is None]
if need:
    print("installation de", need, "...")
    # IMPORTANT : pyctcdecode épingle numpy<2 ; l'installer avec ses dépendances RÉTROGRADE
    # le numpy de Colab et laisse une installation mixte 1.x/2.x -> au prochain import :
    # "numpy.dtype size changed, may indicate binary incompatibility".
    # On installe donc SANS dépendances (pygtrie = sa seule vraie dépendance).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "pyctcdecode", "pygtrie"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jiwer"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "https://github.com/kpu/kenlm/archive/master.zip"], check=False)
    print("⚠️ REDÉMARRE le runtime (Exécution > Redémarrer la session), puis relance cette cellule.")
else:
    import numpy
    print(f"✓ kenlm + pyctcdecode + jiwer disponibles (numpy {numpy.__version__}, intact) — continue en §2")

## 2. CONFIG

In [ ]:
# ============================================================
MODEL_NAME = "baobab-asr-v9-2"
LM_SUBDIR  = "lm_v2"
ALPHA, BETA = 0.7, 0.5              # réglage actuel = baseline de toutes les comparaisons
N_PAR_CASE = 80                     # clips par (langue × type)
MAX_DUR    = 120.0                  # garde-fou durée
# ============================================================

from google.colab import drive
drive.mount("/content/drive")
import os
BASE  = "/content/drive/MyDrive/afrivoices"
MODEL = f"{BASE}/{MODEL_NAME}"
LM    = f"{BASE}/{LM_SUBDIR}"
assert os.path.isdir(MODEL), f"modèle absent: {MODEL}"
assert os.path.isdir(LM), f"LM absent: {LM}"
print(f"modèle: {MODEL_NAME} | LM: {LM_SUBDIR} | baseline (alpha, beta) = ({ALPHA}, {BETA})")

## 3. Modèle + décodeurs + utilitaires (logits direct, secours fp16 si OOM)

In [ ]:
import torch, io, base64, glob, re, contextlib, numpy as np, soundfile as sf, librosa
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor
from pyctcdecode import build_ctcdecoder
from collections import Counter

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "" if device == "cuda" else "⚠️ GPU fortement recommandé")

model = Wav2Vec2BertForCTC.from_pretrained(MODEL, local_files_only=True).to(device).eval()
processor = Wav2Vec2BertProcessor.from_pretrained(MODEL, local_files_only=True)

raw = [t for t, _ in sorted(processor.tokenizer.get_vocab().items(), key=lambda kv: kv[1])]
blank_tok = processor.tokenizer.pad_token
labels, n = [], 0
for t in raw:
    if t == blank_tok: labels.append("")
    elif t in ("|", " "): labels.append(" ")
    elif t in {"[UNK]", "<s>", "</s>"}: n += 1; labels.append("\u2047" * n)
    else: labels.append(t)
assert labels.count("") == 1 and labels.count(" ") == 1

def clean_text(t):
    t = (t or "").lower().strip()
    t = re.sub(r"[^\w\s\u0129\u0169\u00e1\u00e0\u00e2\u00e4\u00e9\u00e8\u00ea\u00eb\u00ed\u00ec\u00ee\u00ef\u00f3\u00f2\u00f4\u00f6\u00fa\u00f9\u00fb\u00fc\']", "", t)
    return re.sub(r"\s+", " ", t)

def duree_bytes(b):
    try: i = sf.info(io.BytesIO(b)); return i.frames / i.samplerate
    except Exception:
        try: i = sf.info(io.BytesIO(base64.b64decode(b))); return i.frames / i.samplerate
        except Exception: return 999.0

def decode_bytes(b):
    try: arr, sr = sf.read(io.BytesIO(b), dtype="float32")
    except Exception: arr, sr = sf.read(io.BytesIO(base64.b64decode(b)), dtype="float32")
    if arr.ndim > 1: arr = arr.mean(axis=1)
    if sr != 16000: arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
    return arr.astype(np.float32)

def unigrams(lang, top=50000):
    c = Counter()
    with open(f"{LM}/{lang}.txt", encoding="utf-8") as f:
        for line in f: c.update(line.split())
    return [w for w, _ in c.most_common(top)]

UNI = {l: unigrams(l) for l in ["swa", "kik", "luo", "kln", "mas", "som"]}
decoders = {l: build_ctcdecoder(labels, kenlm_model_path=f"{LM}/{l}.bin",
                                unigrams=UNI[l], alpha=ALPHA, beta=BETA)
            for l in UNI}
print("✓ 6 décodeurs baseline prêts")

def logits1(arr, fp16=False):
    fe = processor.feature_extractor([arr], sampling_rate=16000, return_tensors="pt")
    ctx = torch.autocast("cuda", dtype=torch.float16) if (fp16 and device == "cuda") else contextlib.nullcontext()
    with torch.no_grad(), ctx:
        return model(**{k: v.to(device) for k, v in fe.items()}).logits[0].float().cpu().numpy()

def logits_robuste(arr):
    """Direct pleine longueur ; si OOM, retry fp16 (autocast)."""
    try:
        return logits1(arr)
    except RuntimeError:
        torch.cuda.empty_cache()
        return logits1(arr, fp16=True)

## 4. Composition EXACTE du test (langue × type) — sans lire l'audio

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

tp = sorted(glob.glob(f"{BASE}/test/**/*.parquet", recursive=True)) or \
     sorted(glob.glob("/content/test_data/**/*.parquet", recursive=True))
if not tp:
    print("test absent localement -> téléchargement HF...")
    from huggingface_hub import snapshot_download
    snapshot_download("digitalumuganda/anv-test-data-nt", repo_type="dataset", local_dir="/content/test_data")
    tp = sorted(glob.glob("/content/test_data/**/*.parquet", recursive=True))
print(f"{len(tp)} parquets test")

parts = []
for p in tqdm(tp, desc="composition", unit="pq"):
    t = pd.read_parquet(p, columns=["language", "type"])
    t["type"] = t["type"].astype(str).str.lower().str.strip()
    parts.append(t)
allt = pd.concat(parts, ignore_index=True)
assert len(allt) == 41733, f"attendu 41733 clips, lu {len(allt)}"
comp = allt.value_counts(["language", "type"]).unstack(fill_value=0)
comp["total"] = comp.sum(axis=1)
comp["p_unscripted"] = comp.get("unscripted", 0) / comp["total"]
print(comp)
tot_u = int(comp.get("unscripted", pd.Series(0, index=comp.index)).sum())
print(f"\nGLOBAL : {tot_u}/41733 = {tot_u/41733:.1%} de clips unscripted")

P_UNSCR = comp["p_unscripted"].to_dict()          # {langue: proportion unscripted du TEST}
POIDS = {(l, "unscripted"): comp.loc[l].get("unscripted", 0) / comp.loc[l, "total"] for l in comp.index}
POIDS.update({(l, "scripted"): 1 - POIDS[(l, "unscripted")] for l in comp.index})
print("\nP_UNSCR =", {k: round(v, 3) for k, v in P_UNSCR.items()})

## 5. Dev STRATIFIÉ par (langue × type) — depuis anvke/ (noms de parquets), toutes durées

swa : pas de brut sur le Drive → case « mixte » depuis `prepared_v5` (le split d'éval
habituel, seed 42). Elle servira aux deux types avec un caveat — swa pèse peu dans la
décision (WER ~0.06), mais garde-le en tête en lisant §7.

In [ ]:
ANVKE = f"{BASE}/anvke"
ALIAS = {"kik": ["kik", "kikuyu", "gikuyu"], "luo": ["luo", "dholuo"], "kln": ["kln", "kalenjin"],
         "mas": ["mas", "maasai"], "som": ["som", "somali"]}
lang_dirs = {}
for d in os.listdir(ANVKE):
    dl = d.lower()
    for lang, als in ALIAS.items():
        if any(a in dl for a in als) and lang not in lang_dirs:
            lang_dirs[lang] = os.path.join(ANVKE, d)
print("langues anvke:", list(lang_dirs))

def remplir(parqs, n):
    pool = []
    for pq in parqs:
        if len(pool) >= n: break
        try:
            df = pd.read_parquet(pq)
        except Exception:
            continue
        df = df.sample(frac=1, random_state=42)
        for _, r in df.iterrows():
            t = (r.get("transcription") or "").strip()
            a = r.get("audio")
            b = a.get("bytes") if isinstance(a, dict) else a
            if not t or b is None: continue
            d = duree_bytes(b)
            if d <= MAX_DUR:
                pool.append((b, clean_text(t), d))
            if len(pool) >= n: break
    return pool

dev = {}   # (lang, type) -> [(bytes, ref, durée)]
for lang, root in lang_dirs.items():
    allp = sorted(glob.glob(f"{root}/**/*.parquet", recursive=True))
    dv = [p for p in allp if "/dev/" in p or os.path.basename(p).startswith("dev")]
    if not dv: dv = allp
    unscr = [p for p in dv if "unscripted" in os.path.basename(p).lower()]
    scr   = [p for p in dv if "unscripted" not in os.path.basename(p).lower()]
    dev[(lang, "scripted")]   = remplir(scr,   N_PAR_CASE)
    dev[(lang, "unscripted")] = remplir(unscr, N_PAR_CASE)

# swa : case mixte depuis prepared_v5 (split d'éval habituel)
try:
    from datasets import load_from_disk
    swa_ds = load_from_disk(f"{BASE}/prepared_v5/swa")
    sp = swa_ds.train_test_split(test_size=60, seed=42)
    rows = []
    for ex in sp["test"]:
        a = ex.get("audio")
        b = a.get("bytes") if isinstance(a, dict) else a
        if b is None: continue
        lab = [t for t in ex.get("labels", []) if t != -100]
        ref = clean_text(processor.tokenizer.decode(lab, group_tokens=False)) if lab else \
              clean_text((ex.get("transcription") or ""))
        if ref: rows.append((b, ref, duree_bytes(b)))
    dev[("swa", "mixte")] = rows[:N_PAR_CASE]
except Exception as ex:
    print("swa indisponible (", str(ex)[:90], ")")

print(f"\n{'case':22} {'n':>4} {'dur.moy':>8} {'dur.max':>8}")
for (lang, typ), pool in sorted(dev.items()):
    ds = [d for _, _, d in pool]
    marge = "" if pool else "  ⚠️ VIDE — vérifie les parquets dev de cette langue"
    print(f"{lang}/{typ:12} {len(pool):>4} " +
          (f"{np.mean(ds):>7.0f}s {max(ds):>7.0f}s" if ds else " " * 17) + marge)

## 6. Cache des logits par case (une passe GPU, direct pleine longueur, secours fp16)

In [ ]:
cache = {}   # (lang, type) -> [(logits fp16, ref)]
for (lang, typ), pool in dev.items():
    lst, skip = [], 0
    for b, ref, d in tqdm(pool, desc=f"{lang}/{typ}", leave=False):
        try:
            arr = decode_bytes(b)
            lst.append((logits_robuste(arr).astype(np.float16), ref))
        except Exception:
            skip += 1
            torch.cuda.empty_cache()
    cache[(lang, typ)] = lst
    torch.cuda.empty_cache()
    print(f"{lang}/{typ}: {len(lst)} logits en cache" + (f" ({skip} clips sautés)" if skip else ""))

## 7. WER par case (baseline 0.7/0.5) + RECALIBRATION avec les poids exacts du test

`WER_test(langue) ≈ p_unscr · WER(unscr) + (1−p_unscr) · WER(scr)` puis macro sur 6 langues.
Le résiduel vs **0.40283** mesure ce qui reste inexpliqué (normalisation du scoring Kaggle,
locuteurs du test — 2 voix pour 818 clips mas scripted ! —, case swa mixte, bruit).

In [ ]:
import jiwer

wer_case = {}
print(f"{'case':22} {'n':>4} | {'WER (0.7,0.5)':>13}")
for (lang, typ), its in sorted(cache.items()):
    if not its: continue
    refs = [r for _, r in its]
    preds = [decoders[lang].decode(lg.astype(np.float32)) for lg, _ in its]
    wer_case[(lang, typ)] = jiwer.wer(refs, preds)
    print(f"{lang}/{typ:12} {len(its):>4} | {wer_case[(lang, typ)]:>13.4f}")

def wer_de(lang, typ):
    if (lang, typ) in wer_case: return wer_case[(lang, typ)]
    if (lang, "mixte") in wer_case: return wer_case[(lang, "mixte")]   # swa
    return None

LB_ANCRE = 0.40283
print(f"\n{'lang':5} {'p_unscr':>8} {'WER_scr':>8} {'WER_uns':>8} | {'WER_test_proj':>13}")
macro, nlang = 0.0, 0
for lang in ["swa", "som", "kik", "luo", "kln", "mas"]:
    p = P_UNSCR.get(lang)
    ws, wu = wer_de(lang, "scripted"), wer_de(lang, "unscripted")
    if p is None or ws is None or wu is None:
        print(f"{lang:5}  — case manquante, exclue du macro"); continue
    proj = p * wu + (1 - p) * ws
    macro += proj; nlang += 1
    tag = "  (mixte->2 types)" if (lang, "mixte") in wer_case else ""
    print(f"{lang:5} {p:>7.1%} {ws:>8.3f} {wu:>8.3f} | {proj:>13.4f}{tag}")
macro /= max(nlang, 1)
print(f"\nMACRO projeté ({nlang} langues) : {macro:.4f}")
print(f"Leaderboard réel               : {LB_ANCRE}")
print(f"Résiduel (scoring Kaggle + locuteurs test + swa mixte + bruit) : {macro - LB_ANCRE:+.4f}")
print("\nSi |résiduel| ≲ 0.03, la calibration est résolue : ce dev stratifié prédit le LB,")
print("et tout gain mesuré ici (grille §8, futures tranches) se transfère en delta direct.")

## 8. Grille (α, β) PAR CASE + comparaison de politiques avec les poids du test

Trois politiques chiffrées **avec les vrais poids** : A = tout-(0.7, 0.5) (actuel),
B = tout-(0.5, 0.0) (la direction systématique vue hier), C = par-case (le meilleur de
chaque case, MAIS seulement si son gain > 0.015 — sinon la case garde (0.7, 0.5)).
La politique gagnante s'écrit telle quelle dans `AB` du notebook de soumission par type.

In [ ]:
import jiwer

A_GRID = [0.3, 0.5, 0.7, 0.9]
B_GRID = [0.0, 0.5, 1.0]
SEUIL_ADOPTION = 0.015

scores = {}   # (lang, typ) -> {(a, b): wer}
for (lang, typ), its in sorted(cache.items()):
    if not its: continue
    refs = [r for _, r in its]
    sc = {}
    for a in A_GRID:
        row = []
        for b in B_GRID:
            dec = build_ctcdecoder(labels, kenlm_model_path=f"{LM}/{lang}.bin",
                                   unigrams=UNI[lang], alpha=a, beta=b)
            sc[(a, b)] = jiwer.wer(refs, [dec.decode(lg.astype(np.float32)) for lg, _ in its])
            row.append(f"b={b}:{sc[(a,b)]:.3f}")
        print(f"{lang}/{typ} a={a}: " + "  ".join(row))
    scores[(lang, typ)] = sc
    best = min(sc, key=sc.get)
    g = sc[(0.7, 0.5)] - sc[best]
    print(f"  -> meilleur {best} = {sc[best]:.4f} | (0.7,0.5) = {sc[(0.7,0.5)]:.4f} | gain {g:+.4f}"
          f" {'→ ADOPTER' if g > SEUIL_ADOPTION else '→ garder (0.7,0.5)'}\n")

def sc_de(lang, typ, ab):
    key = (lang, typ) if (lang, typ) in scores else (lang, "mixte")
    return scores.get(key, {}).get(ab)

def macro_politique(choix):
    """choix : fonction (lang, typ) -> (a, b). Macro pondéré par les poids exacts du test."""
    tot, nl = 0.0, 0
    for lang in ["swa", "som", "kik", "luo", "kln", "mas"]:
        p = P_UNSCR.get(lang)
        wu = sc_de(lang, "unscripted", choix(lang, "unscripted"))
        ws = sc_de(lang, "scripted",   choix(lang, "scripted"))
        if p is None or wu is None or ws is None: continue
        tot += p * wu + (1 - p) * ws; nl += 1
    return tot / max(nl, 1), nl

adopt = {}
for key, sc in scores.items():
    best = min(sc, key=sc.get)
    adopt[key] = best if sc[(0.7, 0.5)] - sc[best] > SEUIL_ADOPTION else (0.7, 0.5)

mA, _ = macro_politique(lambda l, t: (0.7, 0.5))
mB, _ = macro_politique(lambda l, t: (0.5, 0.0))
mC, nl = macro_politique(lambda l, t: adopt.get((l, t), adopt.get((l, "mixte"), (0.7, 0.5))))
print(f"POLITIQUES (macro projeté, {nl} langues, poids exacts du test) :")
print(f"  A. tout-(0.7,0.5)  : {mA:.4f}   (baseline)")
print(f"  B. tout-(0.5,0.0)  : {mB:.4f}   (delta {mB-mA:+.4f})")
print(f"  C. par-case adoptées : {mC:.4f}   (delta {mC-mA:+.4f})")
print("\nRappel : à ~80 clips/case le bruit vaut ±0.01-0.02 par case ; le macro pondéré le")
print("moyenne, mais ne soumets B ou C que si son delta < -0.005.")

print("\nDict AB à coller dans afrivoices_soumission_par_type.ipynb (politique C) :")
LANGS = ["swa", "kik", "luo", "kln", "mas", "som"]
for typ in ["scripted", "unscripted"]:
    ln = {l: adopt.get((l, typ), adopt.get((l, "mixte"), (0.7, 0.5))) for l in LANGS}
    print(f'  "{typ}": {ln},')

## 9. Marche à suivre

1. **§7** : note le résiduel. S'il est petit, ce dev stratifié devient TON instrument —
   toute décision future (politique de décodage, tranches d'entraînement ciblées
   unscripted) se mesure ici avant de dépenser une soumission.
2. **§8** : soumets la politique gagnante avec `afrivoices_soumission_par_type.ipynb`
   (défauts = A ; colle le dict pour B ou C). **Un changement par soumission** : la
   soumission « défauts » (direct + troncature, sans fenêtrage) est déjà utile en soi
   pour vérifier qu'elle retombe ≈ 0.40283 (elle valide le nouveau générateur).
3. La suite au levier principal : tranche d'entraînement pondérée **unscripted**
   (som/kik/kln), avec CE dev stratifié comme critère d'arrêt — on la construit dès que
   §7/§8 sont notés.